In [1]:
import google.generativeai as genai
from collections import defaultdict
import json

/var/folders/f5/j995pwn90q3c6pyc8ts6mh_00000gn/T/ipykernel_5584/2529266802.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [ ]:
genai.configure(api_key="")

In [4]:
aimodel   = "gemini-3.5-flash-lite"

In [5]:
def get_completion_from_messages(messages, system_instruction=None):
    model = genai.GenerativeModel(aimodel, system_instruction=system_instruction)
    response = model.generate_content(
        messages,
        generation_config={"temperature": 0, "max_output_tokens": 10000},
    )
    return response.text

In [6]:
def get_products_and_category():
    """
    Retrieves product details and categorizes them properly.
    Ensures all categories exist even if they have no products.
    """
    predefined_categories = [
        "Computers and Laptops", "Smartphones and Accessories",
        "Televisions and Home Theater Systems", "Gaming Consoles and Accessories",
        "Audio Equipment", "Cameras and Camcorders"
    ]

    try:
        products = get_products()  # Ensure this function is defined and returns a dictionary
        if not isinstance(products, dict):
            raise ValueError("get_products() did not return a valid dictionary.")

        products_by_category = defaultdict(list)

        # Add existing products to their categories
        for product_name, product_info in products.items():
            category = product_info.get('category', None)
            if category:
                products_by_category[category].append(product_name)

        # Ensure all predefined categories exist (even if empty)
        for category in predefined_categories:
            if category not in products_by_category:
                products_by_category[category] = []

        return dict(products_by_category)

    except Exception as e:
        print(f"Error in get_products_and_category: {e}")
        return {category: [] for category in predefined_categories}  # Return empty categories

In [7]:
products_file = "products.json"   # path to your products JSON file (must exist next to this notebook)

def get_products():
    with open(products_file, 'r') as file:
        products = json.load(file)
    return products

In [11]:
def find_category_and_product(user_input, products_and_category):

    delimiter = "####"
    # NOTE: must be an f-string so {delimiter} and {products_and_category} are actually filled in
    system_message = f"""
    You will be provided with customer service queries.
    The customer service query will be delimited with {delimiter} characters.
    Output a python list of json objects, where each object has the following format:
        'category': <one of Computers and Laptops, Smartphones and Accessories, Televisions and Home Theater Systems,
    Gaming Consoles and Accessories, Audio Equipment, Cameras and Camcorders>,
    AND
        'products': <a list of products that must be found in the allowed products below>


    Where the categories and products must be found in the customer service query.
    If a product is mentioned, it must be associated with the correct category in the allowed products list below.
    If no products or categories are found, output an empty list.


    List out all products that are relevant to the customer service query based on how closely it relates
    to the product name and product category.
    Do not assume, from the name of the product, any features or attributes such as relative quality or price.

    The allowed products are provided in JSON format.
    The keys of each item represent the category.
    The values of each item is a list of products that are within that category.
    Allowed products: {products_and_category}"""

    few_shot_user_1 = """I want the most expensive computer."""

    few_shot_assistant_1 = """
    [{'category': 'Computers and Laptops', 'products': ['TechPro Ultrabook', 'BlueWave Gaming Laptop', 'PowerLite Convertible', 'TechPro Desktop', 'BlueWave Chromebook']}]
    """

    # Gemini format: roles are only "user" / "model", content goes in "parts",
    # and the system prompt is passed separately as system_instruction.
    messages = [
        {"role": "user",  "parts": [f"{delimiter}{few_shot_user_1}{delimiter}"]},
        {"role": "model", "parts": [few_shot_assistant_1]},
        {"role": "user",  "parts": [f"{delimiter}{user_input}{delimiter}"]},
    ]

    return get_completion_from_messages(messages, system_instruction=system_message)

In [12]:
products_and_category = get_products_and_category()
products_and_category

{'Computers and Laptops': ['TechPro Ultrabook',
  'BlueWave Gaming Laptop',
  'PowerLite Convertible',
  'TechPro Desktop',
  'BlueWave Chromebook'],
 'Smartphones and Accessories': ['SmartX ProPhone',
  'MobiTech PowerCase',
  'SmartX MiniPhone',
  'MobiTech Wireless Charger',
  'SmartX EarBuds'],
 'Televisions and Home Theater Systems': ['CineView 4K TV',
  'SoundMax Home Theater',
  'CineView 8K TV',
  'SoundMax Soundbar',
  'CineView OLED TV'],
 'Gaming Consoles and Accessories': ['GameSphere X',
  'ProGamer Controller',
  'GameSphere Y',
  'ProGamer Racing Wheel',
  'GameSphere VR Headset'],
 'Audio Equipment': ['AudioPhonic Noise-Canceling Headphones',
  'WaveSound Bluetooth Speaker',
  'AudioPhonic True Wireless Earbuds',
  'WaveSound Soundbar',
  'AudioPhonic Turntable'],
 'Cameras and Camcorders': ['FotoSnap DSLR Camera',
  'ActionCam 4K',
  'FotoSnap Mirrorless Camera',
  'ZoomMaster Camcorder',
  'FotoSnap Instant Camera']}

EVALUATION 1

In [14]:
customer_query1 = """Which TV can I buy if I'm on a budget?"""

products_by_category1 = find_category_and_product(customer_query1, products_and_category)
print(products_by_category1)

[
  {
    "category": "Televisions and Home Theater Systems",
    "products": [
      "CineView 4K TV",
      "SoundMax Home Theater",
      "CineView 8K TV",
      "SoundMax Soundbar",
      "CineView OLED TV"
    ]
  }
]


EVALUATION 2

In [16]:
customer_query2 = """Tell me about the smartx pro phone and the fotosnap camera, the dslr one. Also, what TVs do you have?"""

products_by_category2 = find_category_and_product(customer_query2, products_and_category)
print(products_by_category2)

[
  {
    "category": "Smartphones and Accessories",
    "products": [
      "SmartX ProPhone"
    ]
  },
  {
    "category": "Cameras and Camcorders",
    "products": [
      "FotoSnap DSLR Camera"
    ]
  },
  {
    "category": "Televisions and Home Theater Systems",
    "products": [
      "CineView 4K TV",
      "CineView 8K TV",
      "CineView OLED TV"
    ]
  }
]


EVALUATION 3 - DOES MODEL HALLUCINATE ?

In [17]:
customer_query3 = """Tell me about the CineView TV, the 8K one, Gamesphere console, the X one.
I'm on a budget, what computers do you have?"""

products_by_category3 = find_category_and_product(customer_query3, products_and_category)
print(products_by_category3)

```json
[
  {
    "category": "Televisions and Home Theater Systems",
    "products": [
      "CineView 8K TV"
    ]
  },
  {
    "category": "Gaming Consoles and Accessories",
    "products": [
      "GameSphere X"
    ]
  },
  {
    "category": "Computers and Laptops",
    "products": [
      "TechPro Ultrabook",
      "BlueWave Gaming Laptop",
      "PowerLite Convertible",
      "TechPro Desktop",
      "BlueWave Chromebook"
    ]
  }
]
```


IF MODEL HALLUCINATE, MAKE THE SYSTEM MESSAGE STRONG BY ADDING STRICT CONDITION AND PROVIDING FEW EXAMPLES (FEW SHOT LEARNING)

MORE EVALUATION

In [25]:
def eval_with_rubric(test_set, assistant_answer):

    cust_msg = test_set['customer_msg']
    context = test_set['context']
    completion = assistant_answer
    
    system_message = """\
    You are an assistant that evaluates how well the customer service agent \
    answers a user question by looking at the context that the customer service \
    agent is using to generate its response. 
    """

    user_message = f"""\
You are evaluating a submitted answer to a question based on the context \
that the agent uses to answer the question.
Here is the data:
    [BEGIN DATA]
    ************
    [Question]: {cust_msg}
    ************
    [Context]: {context}
    ************
    [Submission]: {completion}
    ************
    [END DATA]

Compare the factual content of the submitted answer with the context. \
Ignore any differences in style, grammar, or punctuation.
Answer the following questions:
    - Is the Assistant response based only on the context provided? (Y or N)
    - Does the answer include information that is not provided in the context? (Y or N)
    - Is there any disagreement between the response and the context? (Y or N)
    - Count how many questions the user asked. (output a number)
    - For each question that the user asked, is there a corresponding answer to it?
      Question 1: (Y or N)
      Question 2: (Y or N)
      ...
      Question N: (Y or N)
    - Of the number of questions asked, how many of these questions were addressed by the answer? (output a number)
"""

    # Gemini format: "parts" instead of "content", and the system prompt goes in system_instruction
    messages = [
        {'role': 'user', 'parts': [user_message]}
    ]

    response = get_completion_from_messages(messages, system_instruction=system_message)
    return response

In [26]:
customer_msg = """Tell me about the smartx pro phone and the fotosnap camera, the dslr one. 
Also, what TVs or TV related products do you have?"""

cust_prod_info = {
    'customer_msg': customer_msg,
    'context': products_and_category
}

assistant_answer = products_by_category2

evaluation_output = eval_with_rubric(cust_prod_info, assistant_answer)
print(evaluation_output)

- Is the Assistant response based only on the context provided? (Y or N): Y
- Does the answer include information that is not provided in the context? (Y or N): N
- Is there any disagreement between the response and the context? (Y or N): N
- Count how many questions the user asked. (output a number): 2
- For each question that the user asked, is there a corresponding answer to it?
  Question 1: Y
  Question 2: Y
- Of the number of questions asked, how many of these questions were addressed by the answer? (output a number): 2
